# Setup

In [ ]:
# Colab-specific setup
import pathlib

if 'google.colab' not in str(get_ipython()):
    base_folder = pathlib.Path('../../')
else:
    # install extra notebook dependencies in Colab
    ! uv pip install 'watermark==2.4.*' 'pyyaml==6.0.*'

    # mount colab folder
    #from google.colab import drive
    #drive.mount('/content/drive')
    #base_folder = pathlib.Path('/content/drive/MyDrive/Vision/')

    # directly download the prepared dataset and shared helpers
    # (Colab only ever gets the 'small' set -- 'full' is local-only, see 1-preparation.ipynb)
    ! mkdir -p 'results'
    ! wget -nv -P 'prepared' https://raw.githubusercontent.com/mgmalheiros/vision/master/prepared/1-coins-small-image.zip
    ! wget -nv -P 'prepared' https://raw.githubusercontent.com/mgmalheiros/vision/master/prepared/1-coins-small-labels.csv
    ! wget -nv https://raw.githubusercontent.com/mgmalheiros/vision/master/process/counting/common.py
    base_folder = pathlib.Path('./')

In [ ]:
# show library versions
import watermark

# scikit-image also installs imageio and pillow
print(watermark.watermark(packages='skimage,imageio,PIL,pandas,scipy,yaml'))

# Loading

In [ ]:
# 'small' = the 100-image demo set tracked in the repo. 'full' = the 6021-image
# dataset prepared locally by 1-preparation.ipynb -- see the note there. Only
# 'small' exists on Colab, so leave this as 'small' unless running locally.
dataset_size = 'full'  # 'small' or 'full'

import common

images = common.load_prepared_images(base_folder / 'prepared' / f'1-coins-{dataset_size}-image.zip')
df = common.load_labels(base_folder / 'prepared' / f'1-coins-{dataset_size}-labels.csv')

print(f'{len(images)} images loaded')
df.describe()

# Hough Circle Method
Clean up Canny edges (dilate &rarr; fill-holes &rarr; erode, same idea as the edge-based
notebook) and feed the result to a circular Hough transform, which searches a range of
radii for circle-shaped edge patterns directly — no labeling/`regionprops` step needed
here, the peak finder already returns one coordinate per detected coin.

This is by far the most expensive method in this project: it evaluates the Hough
accumulator across every radius in `radius_range` for every image (see the timing in
the Evaluation section below).

In [ ]:
import numpy as np
from scipy import ndimage as ndi
from skimage import color, morphology
from skimage.draw import circle_perimeter
from skimage.feature import canny
from skimage.transform import hough_circle, hough_circle_peaks

In [ ]:
def detect_hough(image, sigma=5, low_threshold=15, high_threshold=35,
                  dilation_disk=4, erosion_disk=14,
                  radius_range=(20, 45), distance_ratio=0.08, peak_threshold_ratio=0.7):
    edges = canny(image, sigma=sigma, low_threshold=low_threshold, high_threshold=high_threshold)
    dilated = morphology.dilation(edges, morphology.disk(dilation_disk))
    filled = ndi.binary_fill_holes(dilated)
    eroded = morphology.erosion(filled, morphology.disk(erosion_disk))

    hough_radii = np.arange(*radius_range)
    hough_res = hough_circle(eroded, hough_radii)

    # search distance scales with image size, so it works across different resolutions
    height, width = image.shape[:2]
    min_distance = int(max(width, height) * distance_ratio)

    accums, cx, cy, radii = hough_circle_peaks(
        hough_res, hough_radii,
        min_xdistance=min_distance, min_ydistance=min_distance,
        threshold=peak_threshold_ratio * np.max(hough_res),
    )
    return len(cx), (cx, cy, radii)

In [ ]:
def draw_circles(image, cx, cy, radii, rgb_color=(220, 20, 20)):
    rgb = color.gray2rgb(image).copy()
    for center_x, center_y, radius in zip(cx, cy, radii):
        circy, circx = circle_perimeter(center_y, center_x, radius, shape=rgb.shape)
        rgb[circy, circx] = rgb_color
    return rgb

## Visual check

In [ ]:
for name in sorted(images)[:3]:
    image = images[name]
    gt = common.real_count(df, name)

    count, (cx, cy, radii) = detect_hough(image)
    result = draw_circles(image, cx, cy, radii)
    common.V(result, f'Real: {gt} | Found: {count}', size=6)

# Evaluation
Run the detector over the whole prepared dataset, score it against `real_count`, and
write a summary to `results/hough_results.yaml` (or `hough_results-full.yaml` when
`dataset_size == 'full'`). Expect this cell to take noticeably longer than the other
notebooks' — see the cost note above.

In [ ]:
results, summary = common.evaluate_method(
    images, df,
    lambda image: detect_hough(image)[0],
    method_name='Hough circle',
    parameters={
        'sigma': 5, 'low_threshold': 15, 'high_threshold': 35,
        'dilation_disk': 4, 'erosion_disk': 14,
        'radius_range': [20, 45], 'distance_ratio': 0.08, 'peak_threshold_ratio': 0.7,
    },
    results_path=base_folder / 'results' / f'hough_results{"" if dataset_size == "small" else "-full"}.yaml',
)

for key, value in summary.items():
    print(f'{key}: {value}')

In [ ]:
results[['abs_error', 'time_seconds', 'peak_memory_kb']].describe()